# 学习的技巧练习（第 5 章）



- 深度神经网络及其问题（梯度消失与梯度爆炸）
- 更新参数方法的优化（SGD 的缺点、Momentum、学习率衰减、AdaGrad、RMSProp、Adam）
- 参数初始化（常数、秩、正态、均匀、Xavier、He）
- 正则化（Batch Normalization、权值衰减、Dropout）
- 应用案例：房价预测

说明：本练习在教材示例的基础上做了适当调整（目标函数、超参数、网络结构等均与教材不同），
请先阅读题目描述，再在下方代码单元格中**手写代码**完成练习，写完后与 `answer` 目录下的答案对照。

部分练习所需的数据位于项目根目录的 `data/` 下，本 notebook 中使用相对路径 `../../data/...`。

先执行下面的单元格导入所需的库。

In [ ]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import matplotlib.pyplot as plt
from torch import optim
from torch.utils.data import TensorDataset, DataLoader

np.set_printoptions(precision=4, suppress=True)
torch.manual_seed(0)

print("torch version:", torch.__version__)

## 5.1 深度神经网络及其问题

网络层数加深后，梯度在隐藏层反向传播时可能"消失"（越来越小）或"爆炸"（越来越大）。
一个直观的解释是：反向传播每经过一层，梯度都会乘上一个系数，连乘的结果决定了梯度最终的尺度。

**练习 1**：用连乘模拟梯度消失与梯度爆炸

反向传播中，梯度每往前一层都会乘上一个近似系数。若系数小于 1，越往前梯度越小（梯度消失）；
若系数大于 1，越往前梯度越大（梯度爆炸）。

要求：

1. 编写函数 `grad_scale(scale, depth)`，返回传播 `depth` 层后的梯度缩放倍数（即 `scale` 的 `depth` 次方），**不要**直接用 `scale ** depth`，请用循环累乘实现；
2. 分别计算 `depth = 10, 20, 50` 时，`scale = 0.5` 与 `scale = 1.5` 的缩放倍数，用科学计数法打印，观察指数级的变化。

In [ ]:
# 练习 1：用连乘模拟梯度消失与梯度爆炸
# TODO: 请在此处手写代码完成练习

**练习 2**：在真实深层网络上观察各层梯度范数

构建一个 8 层隐藏层（每层 32 个神经元）、激活函数为 Sigmoid 的网络，观察反向传播后各层权重梯度的范数
从输出侧到输入侧的变化，体会梯度消失。

要求：

1. 编写 `build_sigmoid_net(depth=8, hidden=32, in_dim=8, std=0.2)`：用 `nn.Sequential` 搭建网络，
   每个隐藏层为 `nn.Linear` + `nn.Sigmoid`，权重用 `nn.init.normal_(mean=0.0, std=std)` 初始化，偏置置 0，
   最后接一个输出维度为 1 的线性层；
2. 用 `x = torch.randn(16, 8)`、`y = torch.randn(16, 1)` 前向计算 MSE 损失并调用 `.backward()`；
3. 遍历 `net.named_parameters()`，对名字中含 `weight` 的参数打印其梯度范数（科学计数法）。

提示：梯度范数用 `p.grad.norm().item()`。

In [ ]:
# 练习 2：在真实深层网络上观察各层梯度范数
# TODO: 请在此处手写代码完成练习

## 5.2 更新参数方法的优化

SGD 简单经典，但在很多问题上并不高效。本节练习 Momentum、学习率衰减以及
AdaGrad / RMSProp / Adam 等改进方法。

为便于对比，我们统一使用教材示例的**修改版**目标函数
$f(x_1,x_2)=0.1x_1^{2}+2x_2^{2}$，其矩阵形式为 `X ** 2 @ w`，其中 `w = torch.tensor([[0.1], [2.0]])`。

**练习 3**：编写通用的优化轨迹记录函数

为了对比不同的参数更新方法，先准备一个工具函数：在给定目标函数上迭代更新自变量 `X`，并记录每一步的 `X`。

要求：编写 `run_optimizer(X, w, optimizer, n_iters)`：

- `X`：初始化好的自变量（`requires_grad=True`），**直接对传入的 `X` 迭代**，不要重新创建；
- 每次迭代：`y = X ** 2 @ w` → `y.backward()` → `optimizer.step()` → `optimizer.zero_grad()`，
  并把当前的 `X` 追加到记录中；
- 返回形状为 `(n_iters + 1, 2)` 的 numpy 数组（含初始点）。

然后用 `w = torch.tensor([[0.1], [2.0]])`、起点 `(-6.0, 1.5)`、`lr = 1e-2`、迭代 300 步，
调用刚写的函数跑一次普通 SGD，打印轨迹数组的形状和最后一个点。

In [ ]:
# 练习 3：编写通用的优化轨迹记录函数
# TODO: 请在此处手写代码完成练习

**练习 4**：SGD 的缺点——学习率不合适导致震荡或发散

目标函数 $f(x_1,x_2)=0.1x_1^{2}+2x_2^{2}$ 的两个方向曲率相差很大（病态），
$x_2$ 方向曲率更大，是最容易引起震荡的方向。

要求：分别用学习率 `0.02`、`0.2`、`0.9`，从 `(-6.0, 1.5)` 出发用 `optim.SGD` 迭代 200 步（复用练习 3 的 `run_optimizer`），
打印每种学习率下的最终点和到最优解 `(0, 0)` 的距离（`np.linalg.norm`），并说明哪种学习率发散。

In [ ]:
# 练习 4：SGD 的缺点：学习率不合适导致震荡或发散
# TODO: 请在此处手写代码完成练习

**练习 5**：Momentum——用动量法减缓震荡、加快收敛

要求：在同一目标函数上，从同一起点 `(-6.0, 1.5)`、同样 `lr = 1e-2`、同样迭代 500 步，
分别用 `optim.SGD`（无动量）与 `optim.SGD(momentum=0.9)` 记录轨迹：

1. 打印两条轨迹的最终点；
2. 在 `(-6, 6) × (-1.5, 1.5)` 范围内的等高线图上，把两条轨迹画出来并加图例（SGD / Momentum）。

In [ ]:
# 练习 5：Momentum：用动量法减缓震荡、加快收敛
# TODO: 请在此处手写代码完成练习

**练习 6**：学习率衰减——等间隔衰减（StepLR）

要求：初始学习率 `lr0 = 0.5`，使用 `optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.6)`，
在目标函数上迭代 90 次。**注意顺序：先 `optimizer.step()`，再 `scheduler.step()`，然后记录学习率**。

1. 记录每一步的 `optimizer.param_groups[0]["lr"]`；
2. 对 `epoch = 0, 14, 15, 29, 30, 45`，用公式 `lr0 * gamma ** ((epoch + 1) // step_size)` 验证记录值是否一致（打印实际值与期望值）；
3. 画出学习率随 epoch 变化的曲线。

In [ ]:
# 练习 6：学习率等间隔衰减（StepLR）
# TODO: 请在此处手写代码完成练习

**练习 7**：学习率衰减——指定间隔衰减（MultiStepLR）

要求：初始学习率 `lr0 = 0.8`，使用 `optim.lr_scheduler.MultiStepLR(optimizer, milestones=[20, 40, 80], gamma=0.5)`，
迭代 100 次，同样先 `optimizer.step()` 再 `scheduler.step()` 并记录学习率。

1. 找出学习率发生变化的 epoch（对比相邻两个记录值）；
2. 对 `epoch = 0, 19, 20, 39, 40, 79, 80`，用"已越过的里程碑个数"作为幂次验证记录值，
   即 `lr0 * gamma ** sum(1 for m in milestones if m <= epoch + 1)`。

In [ ]:
# 练习 7：学习率指定间隔衰减（MultiStepLR）
# TODO: 请在此处手写代码完成练习

**练习 8**：学习率衰减——指数衰减（ExponentialLR）

要求：初始学习率 `lr0 = 0.6`，使用 `optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)`，迭代 60 次，
同样先 `optimizer.step()` 再 `scheduler.step()` 并记录学习率。

1. 对 `epoch = 0, 1, 5, 20`，用公式 `lr0 * gamma ** (epoch + 1)` 验证记录值；
2. 画出学习率随 epoch 变化的曲线（指数曲线的形态）。

In [ ]:
# 练习 8：学习率指数衰减（ExponentialLR）
# TODO: 请在此处手写代码完成练习

**练习 9**：手写 AdaGrad 并与 `torch.optim.Adagrad` 对比

AdaGrad 会为每个参数按历史梯度平方和调整学习率：

$$h \leftarrow h + \nabla^{2}, \qquad W \leftarrow W - \eta\frac{1}{\sqrt{h}}\nabla$$

要求：

1. 编写 `run_adagrad(lr, n_iters, eps=1e-10)`，从 `(-6.0, 1.5)` 出发手写实现上述更新（`h` 初始为 0，
   分母为 `sqrt(h) + eps`），每一步记录 `X`，返回轨迹数组；
2. 编写 `run_torch_opt(cls, lr, n_iters, **kwargs)`，用 `optimizer` 跑同样的迭代并返回轨迹；
3. 用 `lr = 0.5`、迭代 80 步，比较两条轨迹的最大绝对误差，并打印各自的最终点。

提示：手写更新时要在 `torch.no_grad()` 上下文里用 `X -= ...` 原地修改，然后 `X.grad.zero_()`。

In [ ]:
# 练习 9：手写 AdaGrad 并与 torch.optim.Adagrad 对比
# TODO: 请在此处手写代码完成练习

**练习 10**：手写 RMSProp 并与 `torch.optim.RMSprop` 对比

RMSProp 把 AdaGrad 的"全部历史平方和"改为指数移动加权平均：

$$h \leftarrow \alpha h + (1-\alpha)\nabla^{2}, \qquad W \leftarrow W - \eta\frac{1}{\sqrt{h}}\nabla$$

要求：仿照练习 9，编写 `run_rmsprop(lr, n_iters, alpha=0.99, eps=1e-8)` 手写实现，
并用 `optim.RMSprop(lr=0.1, alpha=0.99)`（迭代 100 步）对比两条轨迹的最大绝对误差，打印各自的最终点。

In [ ]:
# 练习 10：手写 RMSProp 并与 torch.optim.RMSprop 对比
# TODO: 请在此处手写代码完成练习

**练习 11**：手写 Adam（含偏差修正）并与 `torch.optim.Adam` 对比

Adam 融合了 Momentum 与 RMSProp，并对一阶、二阶动量做偏差修正：

$$v \leftarrow \alpha_1 v + (1-\alpha_1)\nabla, \quad h \leftarrow \alpha_2 h + (1-\alpha_2)\nabla^{2},
\quad \hat{v}=\frac{v}{1-\alpha_1^{t}}, \quad \hat{h}=\frac{h}{1-\alpha_2^{t}}$$

$$W \leftarrow W - \eta\frac{\hat{v}}{\sqrt{\hat{h}}+\varepsilon}$$

要求：仿照练习 9，编写 `run_adam(lr, n_iters, betas=(0.9, 0.999), eps=1e-8)` 手写实现（`t` 从 1 开始计数，
别忘了偏差修正），并用 `optim.Adam(lr=0.1, betas=(0.9, 0.999))`（迭代 100 步）对比两条轨迹的最大绝对误差。

In [ ]:
# 练习 11：手写 Adam（含偏差修正）并与 torch.optim.Adam 对比
# TODO: 请在此处手写代码完成练习

**练习 12**：多种优化器的对比

要求：在同一起点 `(-6.0, 1.5)`、同一目标函数上，迭代 300 步，对比以下 5 种优化器的收敛情况：

| 优化器 | 参数 |
| --- | --- |
| SGD | `lr=1e-2` |
| Momentum | `lr=1e-2, momentum=0.9` |
| AdaGrad | `lr=0.5` |
| RMSProp | `lr=0.1, alpha=0.99` |
| Adam | `lr=0.1, betas=(0.9, 0.999)` |

1. 用 `run_torch_opt` 记录每种优化器的轨迹，打印各自最终点到最优解的距离；
2. 在等高线图上画出 5 条轨迹并加图例。

提示：可以先用一个字典保存"名字 → (优化器类, 参数)"，再用循环统一处理。

In [ ]:
# 练习 12：多种优化器的对比
# TODO: 请在此处手写代码完成练习

## 5.3 参数初始化

参数初始化的选择对数值稳定性至关重要：初始化不当会导致梯度消失或梯度爆炸。
PyTorch 中可以通过 `nn.init` 下的方法对参数进行初始化。

**练习 13**：常数初始化与秩初始化

要求：

1. 创建一个 `nn.Linear(5, 2)`，依次用 `nn.init.zeros_`、`nn.init.ones_`、`nn.init.constant_(10)`、`nn.init.eye_`
   初始化其权重，每次都把权重打印出来观察；
2. 演示"权重不能全部初始化为同一个值"：创建一个 `nn.Linear(4, 3)`，把权重和偏置全部置 0，
   输入 `torch.randn(2, 4)`，输出求和后反向传播，打印权重的梯度，观察三行梯度是否完全相同。

In [ ]:
# 练习 13：常数初始化与秩初始化
# TODO: 请在此处手写代码完成练习

**练习 14**：正态分布与均匀分布初始化

要求：创建一个 `nn.Linear(1000, 500)`，分别进行下面的初始化，并统计权重的均值、标准差（或用最小/最大值），
与理论值对照：

1. `nn.init.normal_(weight, mean=0.0, std=0.5)`，理论均值 0、标准差 0.5；
2. `nn.init.uniform_(weight, a=-0.05, b=0.05)`，理论区间 $[-0.05, 0.05]$，标准差为 $\frac{b-a}{\sqrt{12}}$。

In [ ]:
# 练习 14：正态分布与均匀分布初始化
# TODO: 请在此处手写代码完成练习

**练习 15**：Xavier 初始化（Glorot 初始化）

Xavier 初始化根据输入数和输出数共同调整权重的初始范围，适用于 Sigmoid / Tanh 等激活函数：

- 正态分布：均值为 0，标准差为 $\sqrt{\frac{2}{n_{in}+n_{out}}}$
- 均匀分布：区间 $\left(-\sqrt{\frac{6}{n_{in}+n_{out}}},\ \sqrt{\frac{6}{n_{in}+n_{out}}}\right)$

要求：对 `nn.Linear(1000, 500)` 分别调用 `nn.init.xavier_normal_` 与 `nn.init.xavier_uniform_`，
用统计出的标准差与理论值对照（正态分布看标准差，均匀分布看绝对值的最大值即边界）。

In [ ]:
# 练习 15：Xavier 初始化
# TODO: 请在此处手写代码完成练习

**练习 16**：He 初始化（Kaiming 初始化）

He 初始化只根据输入数调整权重的初始范围，主要适用于 ReLU 及其变体：

- 正态分布：均值为 0，标准差为 $\sqrt{\frac{2}{n_{in}}}$
- 均匀分布：区间 $\left(-\sqrt{\frac{6}{n_{in}}},\ \sqrt{\frac{6}{n_{in}}}\right)$

要求：对 `nn.Linear(1000, 500)` 分别调用 `nn.init.kaiming_normal_` 与 `nn.init.kaiming_uniform_`，
统计标准差与边界，并与理论值对照。

In [ ]:
# 练习 16：He（Kaiming）初始化
# TODO: 请在此处手写代码完成练习

## 5.4 正则化

过拟合指模型能较好拟合训练数据，却不能很好地预测训练数据之外的数据。常用的正则化方法有
Batch Normalization、权值衰减、Dropout、早停法等。

**练习 17**：BatchNorm1d——手写批量标准化并与 `nn.BatchNorm1d` 对比

Batch Normalization 先对每个特征在 batch 维度上做标准化，再做缩放和平移：

$$\mu=\frac{1}{n}\sum x, \qquad \sigma^{2}=\frac{1}{n}\sum (x-\mu)^{2}, \qquad
\hat{x}=\frac{x-\mu}{\sqrt{\sigma^{2}+\epsilon}}, \qquad y=\gamma\hat{x}+\beta$$

要求：

1. 创建 `nn.BatchNorm1d(num_features=4)`，输入 `torch.randn(6, 4) * 3 + 2`（先 `torch.manual_seed(0)`），
   得到 BN 的输出；
2. 手写实现上述公式（方差用有偏估计，即 `x.var(dim=0, unbiased=False)`；`gamma`、`beta` 用 BN 层自带的
   `weight`、`bias`，`epsilon` 用 `bn.eps`），并与 BN 层的输出比较最大绝对误差；
3. 打印 BN 输出在每个特征方向上的均值，观察是否接近 0。

In [ ]:
# 练习 17：BatchNorm1d：手写批量标准化并与 nn.BatchNorm1d 对比
# TODO: 请在此处手写代码完成练习

**练习 18**：权值衰减

权值衰减通过在损失函数上加入 L2 惩罚项 $L' = L + \frac{1}{2}\lambda\|W\|^{2}$ 抑制过拟合。
惩罚项求导得到 $\lambda W$，因此等价于在梯度上额外加上 $\lambda W$。

要求：给定权重初值 `p0 = torch.tensor([2.0, -1.0])`、梯度 `grad = torch.tensor([0.5, 0.3])`、
`lr = 0.1`、`lam = 0.02`：

1. 手写一步更新：`p0 - lr * (grad + lam * p0)`；
2. 用 `optim.SGD(lr=lr, weight_decay=lam)` 做一步更新（通过 `param.grad = grad` 手动设置梯度）；
3. 比较两者结果是否一致。

In [ ]:
# 练习 18：权值衰减（weight_decay）
# TODO: 请在此处手写代码完成练习

**练习 19**：Dropout——手写随机失活并与 `nn.Dropout` 对比

训练时以概率 $p$ 随机关闭神经元，未被关闭的神经元输出按 $\frac{1}{1-p}$ 缩放，使期望值不变；
测试时不使用 Dropout。

要求：

1. 编写 `my_dropout(x, p, training=True)`：用 `torch.rand_like(x) > p` 生成掩码，返回 `x * mask / (1 - p)`；
   当 `training=False` 或 `p == 0` 时直接返回 `x`；
2. 先 `torch.manual_seed(0)`，用全 1 输入 `torch.ones(10000)`、`p = 0.4`：
   打印手写 Dropout 与 `nn.Dropout(0.4)` 各自的"失活比例"与"非零元素的均值"，并与理论值（0.4、1/(1-0.4)）对照；
3. 验证 `training=False` 时输出与输入完全相同。

In [ ]:
# 练习 19：Dropout：手写随机失活并与 nn.Dropout 对比
# TODO: 请在此处手写代码完成练习

## 5.5 应用案例：房价预测

使用 House Prices 数据集完成一个完整的回归流程：特征工程 → 搭建模型 → 定义损失函数 → 训练模型。

数据位于 `../../data/house_prices.csv`，目标列是 `SalePrice`，其中既有数值型特征也有类别型特征，
并且存在缺失值，需要分别处理。

**练习 20**：特征工程——构造数据集

要求：编写 `create_dataset()` 完成下面的流程，并返回 `(train_dataset, test_dataset, feature_num)`：

1. 用 `pd.read_csv` 读取 `../../data/house_prices.csv`，用 `drop` 去掉无关特征 `Id`；
2. 划分特征 `X`（去掉 `SalePrice`）与目标 `y`（`SalePrice`）；
3. 用 `select_dtypes(exclude="object")` 与 `select_dtypes(include="object")` 分别筛出数值型、类别型特征；
4. 用 `train_test_split` 按 `test_size=0.25, random_state=0` 划分训练集与测试集；
5. 数值型特征：`SimpleImputer(strategy="mean")` 填充缺失值 + `StandardScaler()` 标准化；
   类别型特征：`SimpleImputer(strategy="constant", fill_value="NaN")` 填充 + `OneHotEncoder(handle_unknown="ignore")` 独热编码；
   两者用 `ColumnTransformer` 组合，`fit_transform` 训练集、`transform` 测试集（注意转成 `DataFrame` 并取列名）；
6. 用 `TensorDataset` 包装成 `(float32 特征, float32 目标)`，并返回特征数量。

最后调用一次，打印特征数量与训练/测试集大小。

In [ ]:
# 练习 20：特征工程：构造数据集
# TODO: 请在此处手写代码完成练习

**练习 21**：搭建模型与损失函数

要求：

1. 用 `nn.Sequential` 搭建一个两层隐藏层的回归模型，结构为
   `Linear(feature_num, 256) → BatchNorm1d(256) → ReLU → Dropout(0.3) → Linear(256, 64) → BatchNorm1d(64) → ReLU → Dropout(0.3) → Linear(64, 1)`，并打印模型结构；
2. 实现对数均方根误差损失（房价预测更关心相对误差）：

$$Loss=\sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(\log(\hat{y})-\log(y)\right)^{2}}$$

   注意把预测值 `squeeze` 后用 `torch.clamp(pred, 1, float("inf"))` 限制在 1 到正无穷之间（避免 `log` 出现非法值）；
3. 用 `pred = torch.tensor([100.0, 200.0])`、`target = torch.tensor([110.0, 180.0])` 验证损失函数能正常计算。

In [ ]:
# 练习 21：搭建模型与损失函数
# TODO: 请在此处手写代码完成练习

**练习 22**：模型训练与损失曲线

要求：编写 `train(model, train_dataset, test_dataset, lr, epoch_num, batch_size, device)`：

1. 用一个 `init_weight(layer)` 函数对线性层做 `nn.init.kaiming_normal_(layer.weight)` 初始化，
   通过 `model.apply(init_weight)` 应用；
2. 优化器使用 `torch.optim.Adam(model.parameters(), lr=lr)`；
3. 每个 epoch：`model.train()` + `DataLoader(shuffle=True)` 训练，累加并记录平均 `log_rmse`；
   然后 `model.eval()` + `torch.no_grad()` 在测试集上（`shuffle=False`）评估并记录平均 `log_rmse`；
4. 每个 epoch 打印一次 train / test 损失，最后返回两个损失列表；
5. 用 `lr=0.05`、`epoch_num=30`、`batch_size=64` 训练，并把两条损失曲线画在同一张图上。

提示：设备用 `torch.device("cuda" if torch.cuda.is_available() else "cpu")`。

In [ ]:
# 练习 22：模型训练与损失曲线
# TODO: 请在此处手写代码完成练习